# 08 -- Monte Carlo risk analysis

Paired script: `analysis/monte_carlo.py`. Seeded bootstrap resampling
(`analysis.resampling.seeded_bootstrap_indices`) of a trade P/L sequence to characterize
the distribution of final balance, max drawdown, and probability of hitting a ruin
threshold. **This is a FIXED-CASH model, not a percentage-of-equity compounding model** --
see the script's own module docstring for why that distinction matters for this project's
percentage-risk sizing.

**Uses clearly-labelled SYNTHETIC P/L data.** Real-data run: PENDING.

In [1]:
import sys
import tempfile
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from analysis.monte_carlo import run

In [2]:
tmp_dir = Path(tempfile.mkdtemp(prefix="themba_montecarlo_demo_"))
# **Bumped to 20 trades, 2026-07-22 Codex review finding:** a MIN_N_TRADES
# floor was added -- resampling a single (or tiny) historical trade
# manufactures apparent precision the underlying sample cannot support.
pnl_pattern = [30.0, -20.0, 15.0, -10.0, 25.0, -15.0]
pd.DataFrame(
    {
        "trade_id": [f"t{i}" for i in range(20)],
        "profit": (pnl_pattern * 4)[:20],
    }
).to_csv(tmp_dir / "trades.csv", index=False)

In [3]:
result = run(
    tmp_dir / "trades.csv",
    n_resamples=2000,
    seed=42,
    starting_balance=1000.0,
    ruin_threshold=500.0,
    output_json=tmp_dir / "mc.json",
    repo_path=PROJECT_ROOT.parents[1],
)

# **Terminology fixed, 2026-07-22 Codex review finding:** these are PERCENTILE
# SCENARIO BOUNDS from the i.i.d. empirical-bootstrap resampling procedure
# (see monte_carlo.py's own module docstring), not a conventional statistical
# confidence interval for a future balance/drawdown -- they describe the
# spread of outcomes THIS resampling procedure produces from THIS historical
# sample, destroying any real streak/autocorrelation/cooldown structure.
print(f"final_balance_mean            = {result.final_balance_mean:.2f}")
print(
    f"final_balance_scenario_bounds = [{result.final_balance_ci_lower:.2f}, {result.final_balance_ci_upper:.2f}]"
)
print(f"max_drawdown_pct_mean          = {result.max_drawdown_pct_mean:.4f}")
print(f"prob_ruin (<=500)              = {result.prob_ruin}")
print(
    f"prob_ruin_finite_sim_ci        = [{result.prob_ruin_ci_lower:.4f}, {result.prob_ruin_ci_upper:.4f}]"
)

# Determinism check: re-running with the identical seed reproduces the exact result.
result_repeat = run(
    tmp_dir / "trades.csv", n_resamples=2000, seed=42, starting_balance=1000.0, ruin_threshold=500.0
)
assert result == result_repeat

final_balance_mean            = 1090.50
final_balance_scenario_bounds = [920.00, 1265.00]
max_drawdown_pct_mean          = 0.0601
prob_ruin (<=500)              = 0.0
prob_ruin_finite_sim_ci        = [0.0000, 0.0019]


## Real-data run: PENDING

Requires real trade P/L history -- none exists yet.